# Make a slim copy of the baseline weights

The baseline run saved `final.weights.h5` at about 1.58 GB because the file also holds the optimizer state. This notebook loads it with the saved architecture and re-saves **weights only** (about 525 MB), which is all the simulator needs. Nothing is retrained.

**Before running:** click *Add Input* and add the output of your `baseline 60 -> 224` notebook.

In [ ]:
import copy, glob, json, os, shutil
import numpy as np
import keras

# Find final.weights.h5 that sits next to model_architecture.json (the baseline run folder)
candidates = [
    p for p in glob.glob('/kaggle/input/**/final.weights.h5', recursive=True)
    if os.path.exists(os.path.join(os.path.dirname(p), 'model_architecture.json'))
]
print('Found:', candidates)
assert len(candidates) >= 1, 'Add the baseline notebook output as an input first.'
weights_path = candidates[0]
folder = os.path.dirname(weights_path)
print('Using folder:', folder)
print('Original file size (GB): %.2f' % (os.path.getsize(weights_path) / 1e9))

In [ ]:
config = json.load(open(os.path.join(folder, 'model_architecture.json')))['config']
model = keras.models.Model.from_config(copy.deepcopy(config))
model.load_weights(weights_path)
print('Parameters:', f'{model.count_params():,}', '(expected 131,322,947)')
assert model.count_params() == 131322947

In [ ]:
out_dir = '/kaggle/working/baseline'
os.makedirs(out_dir, exist_ok=True)
slim_path = os.path.join(out_dir, 'final.weights.h5')
model.save_weights(slim_path)
shutil.copy(os.path.join(folder, 'model_architecture.json'), os.path.join(out_dir, 'config.json'))
print('Slim file size (MB): %.1f' % (os.path.getsize(slim_path) / 1e6))

In [ ]:
# Check: a fresh model loaded from the slim file must give identical predictions
fresh = keras.models.Model.from_config(copy.deepcopy(config))
fresh.load_weights(slim_path)
x = np.random.default_rng(0).random((2, 224, 224, 3)).astype('float32')
print('Original :', np.round(model.predict(x, verbose=0), 6).tolist())
print('Slim file:', np.round(fresh.predict(x, verbose=0), 6).tolist())
print('Max difference:', float(np.max(np.abs(model.predict(x, verbose=0) - fresh.predict(x, verbose=0)))))

**Next:** open this notebook's *Output* tab and download `baseline/final.weights.h5` (about 525 MB). Rename it as the README says and place it in `server/model/baseline/`.